# Gold — `dbr_caresync.gold.hospital_daily_summary`

One row per hospital per calendar day. Aggregates lab_results by hospital and date, joins to hospitals and calendar dimension. Includes LAG-based day-over-day test volume change. Full rebuild on every run.

In [0]:
# ── Ensure gold schema exists ─────────────────────────────────────────────────
spark.sql("CREATE SCHEMA IF NOT EXISTS dbr_caresync.gold")

# ── Build gold.hospital_daily_summary ───────────────────────────────────────────
spark.sql("""
CREATE OR REPLACE TABLE dbr_caresync.gold.hospital_daily_summary AS
WITH base AS (
    SELECT
        lr.hospital_id,
        lr.test_date                                                               AS summary_date,
        COUNT(*)                                                                   AS total_tests,
        COUNT(DISTINCT lr.patient_id)                                              AS unique_patients_tested,
        COUNT(DISTINCT lr.doctor_id)                                               AS active_doctor_count,
        COUNT(CASE WHEN lr.result_status = 'Abnormal' THEN 1 END)                 AS abnormal_test_count,
        COUNT(CASE WHEN lr.result_status = 'Critical' THEN 1 END)                 AS critical_test_count
    FROM dbr_caresync.silver.lab_results lr
    GROUP BY lr.hospital_id, lr.test_date
),
with_window AS (
    SELECT
        b.hospital_id,
        h.hospital_name,
        b.summary_date,
        c.day_name,
        c.is_weekend,
        b.total_tests,
        b.unique_patients_tested,
        b.active_doctor_count,
        b.abnormal_test_count,
        b.critical_test_count,
        CAST(b.abnormal_test_count * 100.0 / b.total_tests AS DECIMAL(5, 2))     AS abnormal_rate_pct,
        CAST(b.critical_test_count * 100.0 / b.total_tests AS DECIMAL(5, 2))     AS critical_rate_pct,
        LAG(b.total_tests) OVER (
            PARTITION BY b.hospital_id ORDER BY b.summary_date
        )                                                                          AS prev_day_total_tests
    FROM base b
    LEFT JOIN dbr_caresync.silver.hospitals h
        ON b.hospital_id = h.hospital_id
    LEFT JOIN dbr_caresync.silver.calendar c
        ON CAST(DATE_FORMAT(b.summary_date, 'yyyyMMdd') AS INT) = c.date_key
)
SELECT
    hospital_id,
    hospital_name,
    summary_date,
    day_name,
    is_weekend,
    total_tests,
    unique_patients_tested,
    active_doctor_count,
    abnormal_test_count,
    critical_test_count,
    abnormal_rate_pct,
    critical_rate_pct,
    prev_day_total_tests,
    CASE
        WHEN prev_day_total_tests IS NULL OR prev_day_total_tests = 0 THEN NULL
        ELSE CAST((total_tests - prev_day_total_tests) * 100.0 / prev_day_total_tests AS DECIMAL(6, 2))
    END AS test_volume_change_pct
FROM with_window
""")

print("\u2705 Table created: dbr_caresync.gold.hospital_daily_summary")
result = spark.table("dbr_caresync.gold.hospital_daily_summary")
print(f"   Rows    : {result.count():,}")
print(f"   Columns : {len(result.columns)}")
display(result.limit(5))

In [0]:
record_count = result.count()
dbutils.notebook.exit(str(record_count))